# ✈️ Airline Route Profitability & Cost Analysis
## A Comprehensive Financial, Operational, and Machine Learning Study of a Dubai-Hub Carrier

---

This notebook presents an end-to-end analytical study of flight-level financial data for a major Middle Eastern airline operating out of **Dubai International Airport (DXB)**. The dataset covers approximately **7,500 flight records across 30 routes**, spanning short-haul, medium-haul, and long-haul categories over a full calendar year of operations.

The analysis is structured into **ten major sections**:

| # | Section | Description |
|---|---------|-------------|
| 1 | **Environment Setup** | Library imports, global configuration, colour palettes |
| 2 | **Data Quality & Cleaning** | Completeness checks, type validation, logical consistency |
| 3 | **Exploratory Data Analysis** | Fleet overview, financial distributions, route rankings, correlation |
| 4 | **Cost Structure Decomposition** | 14-component cost mix, CASK/RASK efficiency metrics |
| 5 | **Seasonality & Demand Analysis** | Monthly trends, seasonal load factor, route-season heatmaps |
| 6 | **Regression Modelling** | 7-model benchmark for predicting profit margin, SHAP interpretation |
| 7 | **Classification Modelling** | 5-model benchmark for predicting route profitability (binary) |
| 8 | **Route Clustering & Segmentation** | K-Means with elbow/silhouette selection, PCA visualisation |
| 9 | **What-If Scenario Simulator** | Interactive planning tool for load-factor and season scenarios |
| 10 | **Conclusions & Recommendations** | Consolidated findings and strategic actions |

### Objectives
- Understand which routes, aircraft types, and seasons drive profitability
- Decompose the cost structure to identify the largest controllable levers
- Train and compare **multiple ML models** for both regression and classification tasks
- Use SHAP to transform black-box predictions into interpretable business insights
- Provide a scenario simulator for scheduling and pricing decisions

### Dataset at a Glance
| Attribute | Detail |
|-----------|--------|
| **File** | `airline_route_profitability.csv` |
| **Rows** | ~7,500 individual flight records |
| **Routes** | 30 destinations from DXB |
| **Period** | Full calendar year (Jan–Dec) |
| **Target (Regression)** | `Profit_Margin` (%) |
| **Target (Classification)** | `Is_Profitable` (binary) |
| **Key Features** | Aircraft type, passengers, load factor, costs, season, demand level |

> **Reproducibility note:** All random seeds are fixed at `RANDOM_STATE = 42`. Run all cells top-to-bottom for a fully deterministic execution.


---
## 1. Environment Setup and Data Loading

### 1.1 Library Imports

We import all libraries up-front so any missing dependency surfaces immediately rather than mid-analysis.  
Key packages and their roles:

| Library | Role |
|---------|------|
| `pandas` / `numpy` | Data wrangling and numerical computation |
| `matplotlib` / `seaborn` | Static visualisation |
| `scikit-learn` | Preprocessing, classical ML models, evaluation metrics |
| `lightgbm` / `xgboost` | Gradient-boosted tree ensembles (often best-in-class on tabular data) |
| `catboost` | Gradient boosting with native categorical handling |
| `shap` | Model-agnostic explainability via Shapley values |
| `scipy` | Statistical tests (Mann-Whitney U, Kruskal-Wallis) |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from scipy import stats
from itertools import combinations

# ── Classical ML ──────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, StratifiedKFold,
    GridSearchCV
)
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# ── Regression models ──────────────────────────────────────────────────────
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesRegressor, ExtraTreesClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# ── Boosting libraries ─────────────────────────────────────────────────────
import lightgbm as lgb
import xgboost as xgb

try:
    from catboost import CatBoostRegressor, CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("CatBoost not installed – will skip CatBoost models.")

import shap

warnings.filterwarnings('ignore')

# ── Global plot configuration ──────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'figure.titleweight': 'bold',
})

# ── Colour palettes ────────────────────────────────────────────────────────
PALETTE_ROUTE  = {'Short Haul': '#2196F3', 'Medium Haul': '#FF9800', 'Long Haul': '#4CAF50'}
PALETTE_SEASON = {'Peak': '#E53935', 'Shoulder': '#FB8C00', 'Normal': '#43A047', 'Low': '#1E88E5'}

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries loaded successfully.")
print(f"  Pandas   : {pd.__version__}")
print(f"  LightGBM : {lgb.__version__}")
print(f"  XGBoost  : {xgb.__version__}")
print(f"  SHAP     : {shap.__version__}")
print(f"  CatBoost : {'available' if CATBOOST_AVAILABLE else 'not installed'}")


### 1.2 Data Loading

We load the CSV, display its shape and memory footprint, and preview the first few rows to confirm that the file loaded correctly.


In [ ]:
DATA_PATH = '/kaggle/input/airline-route-profitability-and-cost-analysis/airline_route_profitability.csv'

df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset shape   : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Memory usage    : {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()
df_raw.head()


---
## 2. Data Quality Assessment and Cleaning

Trustworthy analysis starts with trustworthy data. We run four classes of checks before touching a single chart:

1. **Completeness** — Are there missing values? If so, how many and in which columns?  
2. **Duplicates** — Do exact duplicate rows exist? (A common artefact of data pipeline joins.)  
3. **Type correctness** — Are dates parsed as dates, numbers stored as floats, categoricals as strings?  
4. **Logical consistency** — Do derived fields (`Load_Factor`, `Profit`, `Profit_Margin`) agree with their component columns? Discrepancies here signal upstream calculation errors or rounding issues.

Any failures trigger a correction step before the analysis proceeds.


In [ ]:
print("=== Column Data Types ===")
print(df_raw.dtypes)
print()

print("=== Missing Values ===")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing Pct (%)': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0] if missing_df['Missing Count'].sum() > 0
      else "✓ No missing values detected.")


In [ ]:
print("=== Duplicate Records ===")
n_dupes = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}" + (" ✓" if n_dupes == 0 else " ⚠️  – will be dropped"))

print()
print("=== Numerical Summary ===")
df_raw.describe(include='number').T.round(2)


In [ ]:
# ── Build working copy ────────────────────────────────────────────────────
df = df_raw.copy()
if df.duplicated().sum() > 0:
    df = df.drop_duplicates()
    print(f"Dropped {df_raw.shape[0] - df.shape[0]} duplicate rows.")

df['Flight_Date'] = pd.to_datetime(df['Flight_Date'])
df['Month']       = df['Flight_Date'].dt.month
df['Month_Name']  = df['Flight_Date'].dt.strftime('%b')
df['DayOfWeek']   = df['Flight_Date'].dt.day_name()
df['Quarter']     = df['Flight_Date'].dt.quarter

# ── Logical consistency checks ─────────────────────────────────────────────
df['Load_Factor_Computed']  = df['Passengers'] / df['Aircraft_Capacity']
df['Profit_Computed']       = df['Total_Revenue'] - df['Total_Cost']
df['Profit_Margin_Computed'] = (df['Profit'] / df['Total_Revenue'] * 100).round(2)

checks = {
    'Load_Factor vs Passengers/Capacity': (df['Load_Factor'] - df['Load_Factor_Computed']).abs().max(),
    'Profit vs Revenue-Cost':             (df['Profit'] - df['Profit_Computed']).abs().max(),
    'Profit_Margin vs Profit/Revenue*100':(df['Profit_Margin'] - df['Profit_Margin_Computed']).abs().max(),
}

for description, max_diff in checks.items():
    status = "✓" if max_diff < 0.01 else "⚠️ "
    print(f"  {status}  {description}: max diff = {max_diff:.6f}")

print()
print("Data integrity checks complete. Proceeding with feature engineering.")


In [ ]:
# ── Feature Engineering ────────────────────────────────────────────────────
# Cost column groups (used throughout the notebook)
DIRECT_COSTS   = ['Fuel_Cost', 'Maintenance_Cost', 'Crew_Cost', 'Depreciation_Cost', 'Insurance_Cost']
SERVICE_COSTS  = ['Airport_Fees', 'Catering_Cost', 'Handling_Cost', 'Navigation_Fees']
INDIRECT_COSTS = ['Sales_Distribution_Cost', 'Passenger_Service_Cost',
                  'Overhead_Cost', 'Marketing_Cost', 'IT_Systems_Cost']
ALL_COST_COLS  = DIRECT_COSTS + SERVICE_COSTS + INDIRECT_COSTS

# Revenue and cost efficiency features
df['Revenue_Per_Passenger'] = df['Total_Revenue'] / df['Passengers']
df['Cost_Per_Passenger']    = df['Total_Cost']    / df['Passengers']
df['Fuel_Pct_Total_Cost']   = df['Fuel_Cost']     / df['Total_Cost'] * 100
df['Is_Profitable']         = (df['Profit'] > 0).astype(int)

# CASK / RASK — the airline industry's unit cost and unit revenue benchmarks
# Distance is approximated as Flight_Hours × 850 km/h (typical cruise speed)
df['Estimated_Distance_km'] = df['Flight_Hours'] * 850
df['CASK'] = df['Total_Cost']    / (df['Aircraft_Capacity'] * df['Estimated_Distance_km'])
df['RASK'] = df['Total_Revenue'] / (df['Aircraft_Capacity'] * df['Estimated_Distance_km'])

# Yield: revenue per revenue passenger kilometre (RPK)
df['Yield'] = df['Total_Revenue'] / (df['Passengers'] * df['Estimated_Distance_km'])

# Cost index: how much of each dollar of revenue is consumed by cost
df['Cost_Revenue_Ratio'] = df['Total_Cost'] / df['Total_Revenue']

# Flight efficiency: passengers per flight hour
df['Pax_Per_Flight_Hour'] = df['Passengers'] / df['Flight_Hours']

print("Feature engineering complete.")
print(f"Working dataframe shape: {df.shape}")
print(f"Engineered features added: Revenue_Per_Passenger, Cost_Per_Passenger, Fuel_Pct_Total_Cost,")
print(f"  Is_Profitable, CASK, RASK, Yield, Cost_Revenue_Ratio, Pax_Per_Flight_Hour")


---
## 3. Exploratory Data Analysis

Exploratory analysis answers the "what" before modelling answers the "why". We examine the distribution of every major financial metric, identify which routes and aircraft types perform best, and quantify relationships between variables through correlation analysis.

The subsections progress from broad (fleet-level overview) to specific (pairwise correlations and statistical significance tests).


### 3.1 Fleet and Route Network Overview

The first question any airline analyst asks is: *What does the operation look like?* We visualise the mix of aircraft types, the share of flights by route category, and the most-served destinations. This contextualises all subsequent profitability analysis — a network skewed toward short-haul will behave very differently from one dominated by long-haul.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Fleet and Route Network Overview')

ac_counts = df['Aircraft_Type'].value_counts()
sns.barplot(x=ac_counts.values, y=ac_counts.index, palette='Blues_d', ax=axes[0])
axes[0].set_title('Flights by Aircraft Type')
axes[0].set_xlabel('Number of Flights')
axes[0].set_ylabel('')
for bar, val in zip(axes[0].patches, ac_counts.values):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)

rc_counts = df['Route_Category'].value_counts()
axes[1].pie(rc_counts.values, labels=rc_counts.index,
            colors=[PALETTE_ROUTE[c] for c in rc_counts.index],
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 10})
axes[1].set_title('Flights by Route Category')

dest_counts = df['Destination'].value_counts().head(15)
sns.barplot(x=dest_counts.values, y=dest_counts.index, palette='Oranges_d', ax=axes[2])
axes[2].set_title('Top 15 Destinations by Flight Count')
axes[2].set_xlabel('Number of Flights')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"Route breakdown:")
for cat, cnt in rc_counts.items():
    print(f"  {cat:15s}: {cnt:,} flights ({cnt/len(df)*100:.1f}%)")


### 3.2 Financial Metric Distributions

Before computing any aggregate statistics, we inspect the full shape of each financial distribution. Key questions:
- **Skewness**: Are revenues and profits approximately normally distributed, or do a few high-value routes dominate?
- **Bimodality**: Does the profit margin distribution show two modes? This would suggest a structural divide between route types.
- **Outliers**: Are extreme values genuine (a cancelled flight, a bulk-booking event) or data errors?

Median lines are shown on each histogram to allow a quick comparison with the mean and detect distributional skew.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distributions of Key Financial Metrics')

metrics = [
    ('Total_Revenue',         'Total Revenue (USD)',            'steelblue'),
    ('Total_Cost',            'Total Cost (USD)',               'coral'),
    ('Profit',                'Profit (USD)',                   'mediumseagreen'),
    ('Profit_Margin',         'Profit Margin (%)',              'mediumpurple'),
    ('Load_Factor',           'Load Factor',                   'goldenrod'),
    ('Revenue_Per_Passenger', 'Revenue Per Passenger (USD)',    'teal'),
]

for ax, (col, label, color) in zip(axes.flat, metrics):
    sns.histplot(df[col], bins=50, kde=True, color=color, ax=ax, edgecolor='none', alpha=0.75)
    med = df[col].median()
    mn  = df[col].mean()
    ax.axvline(med, color='black',  linestyle='--', linewidth=1.2, label=f'Median: {med:,.1f}')
    ax.axvline(mn,  color='orange', linestyle=':',  linewidth=1.2, label=f'Mean:   {mn:,.1f}')
    ax.set_title(label)
    ax.set_xlabel('')
    ax.legend(fontsize=8)
    skew = df[col].skew()
    ax.text(0.97, 0.95, f'Skew: {skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()


### 3.3 Route Profitability Rankings

We aggregate profit margin to the route level and rank all 30 destinations. A dual-panel chart shows the top 15 and bottom 15 simultaneously, making structural winners and losers immediately apparent. The colour coding (blue/orange/green for short/medium/long haul) exposes whether profitability differences are driven by route length or route-specific factors.

**Statistical note:** We also compute the percentage of individual flights that are profitable on each route, which can diverge from the average margin when a route has high variance (occasional very profitable and very unprofitable flights).


In [ ]:
route_summary = df.groupby(['Destination', 'Route_Category']).agg(
    Flights            = ('Flight_Number',    'count'),
    Avg_Profit_Margin  = ('Profit_Margin',    'mean'),
    Median_Profit_Margin = ('Profit_Margin',  'median'),
    Std_Profit_Margin  = ('Profit_Margin',    'std'),
    Total_Profit       = ('Profit',           'sum'),
    Total_Revenue      = ('Total_Revenue',    'sum'),
    Avg_Load_Factor    = ('Load_Factor',      'mean'),
    Pct_Profitable     = ('Is_Profitable',    'mean'),
    Avg_CASK           = ('CASK',             'mean'),
    Avg_RASK           = ('RASK',             'mean'),
).reset_index().sort_values('Avg_Profit_Margin', ascending=False)

route_summary['Pct_Profitable'] = route_summary['Pct_Profitable'] * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle('Route Profitability Rankings — Average Profit Margin by Destination')

for ax, df_slice, title in [
    (axes[0], route_summary.head(15), 'Top 15 Most Profitable Routes'),
    (axes[1], route_summary.tail(15).sort_values('Avg_Profit_Margin'), 'Bottom 15 Least Profitable Routes'),
]:
    colors = [PALETTE_ROUTE[c] for c in df_slice['Route_Category']]
    bars = ax.barh(df_slice['Destination'], df_slice['Avg_Profit_Margin'], color=colors)
    ax.set_title(title)
    ax.set_xlabel('Average Profit Margin (%)')
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=1)
    for bar, val in zip(bars, df_slice['Avg_Profit_Margin']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

legend_elements = [mpatches.Patch(facecolor=v, label=k) for k, v in PALETTE_ROUTE.items()]
for ax in axes:
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

print("Full route summary (all 30 destinations):")
route_summary.round(2).to_string(index=False)


### 3.4 Profit Margin by Route Category and Aircraft Type

Violin plots convey both the median and the full probability density of profit margin within each segment — substantially more informative than simple box plots for understanding whether distributions are unimodal, bimodal, or skewed. Overlaid quartile lines preserve the five-number summary for rapid comparison.

**Heatmap:** The aircraft × route category median margin matrix identifies which pairings are most and least efficient, informing fleet deployment decisions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Profit Margin Distribution by Operational Segment')

order_cat = ['Short Haul', 'Medium Haul', 'Long Haul']
sns.violinplot(data=df, x='Route_Category', y='Profit_Margin',
               order=order_cat, palette=PALETTE_ROUTE,
               inner='quartile', cut=0, ax=axes[0])
axes[0].set_title('Profit Margin by Route Category\n(Violin = density, lines = quartiles)')
axes[0].set_xlabel('Route Category')
axes[0].set_ylabel('Profit Margin (%)')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.2, label='Break-Even')
axes[0].legend()

# Add mean markers
for i, cat in enumerate(order_cat):
    mn = df[df['Route_Category'] == cat]['Profit_Margin'].mean()
    axes[0].scatter(i, mn, color='black', s=60, zorder=5, marker='D', label='Mean' if i == 0 else '')
axes[0].legend()

ac_order = df.groupby('Aircraft_Type')['Profit_Margin'].median().sort_values(ascending=False).index
sns.violinplot(data=df, x='Aircraft_Type', y='Profit_Margin',
               order=ac_order, palette='Set2', inner='quartile', cut=0, ax=axes[1])
axes[1].set_title('Profit Margin by Aircraft Type\n(Violin = density, lines = quartiles)')
axes[1].set_xlabel('Aircraft Type')
axes[1].set_ylabel('Profit Margin (%)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.2)

plt.tight_layout()
plt.show()


In [ ]:
# Kruskal-Wallis test: is the difference in profit margin across route categories statistically significant?
groups = [df[df['Route_Category'] == cat]['Profit_Margin'].values for cat in order_cat]
stat, pval = stats.kruskal(*groups)
print(f"Kruskal-Wallis test across route categories: H = {stat:.2f}, p-value = {pval:.4e}")
print("  → The difference in profit margins across route categories is",
      "statistically significant (p < 0.05)." if pval < 0.05 else "NOT statistically significant.")
print()

# Pairwise Mann-Whitney U tests
print("Pairwise Mann-Whitney U tests:")
for a, b in combinations(order_cat, 2):
    u, p = stats.mannwhitneyu(df[df['Route_Category'] == a]['Profit_Margin'],
                               df[df['Route_Category'] == b]['Profit_Margin'],
                               alternative='two-sided')
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"  {a:15s} vs {b:15s}: U = {u:.0f}, p = {p:.4e}  {sig}")


In [ ]:
# Heatmap: Median Profit Margin by Aircraft Type × Route Category
pivot_ac_route = df.pivot_table(
    index='Aircraft_Type', columns='Route_Category',
    values='Profit_Margin', aggfunc='median'
)[['Short Haul', 'Medium Haul', 'Long Haul']]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_ac_route, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, cbar_kws={'label': 'Median Profit Margin (%)'},
            ax=ax, vmin=-20)
ax.set_title('Median Profit Margin (%) — Aircraft Type × Route Category\n'
             '(Green = higher margin  |  Red = lower margin)')
ax.set_xlabel('Route Category')
ax.set_ylabel('Aircraft Type')
plt.tight_layout()
plt.show()
print("\nInsight: This matrix immediately reveals which aircraft-route pairings are most and least")
print("economically efficient. Rows/columns with consistent red cells are candidates for review.")


### 3.5 Load Factor Analysis

Load factor — the fraction of available seats that are occupied — is the most widely tracked demand metric in aviation. Its relationship with profitability is non-linear: once load factor exceeds the break-even threshold (typically 65–75% for the cost structure of a given route), each additional passenger is almost pure margin. Below that threshold, fixed costs overwhelm revenue.

We quantify the correlation, stratify by route category, and identify which destinations consistently under-fill their aircraft.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Load Factor Analysis')

for cat, color in PALETTE_ROUTE.items():
    subset = df[df['Route_Category'] == cat]
    axes[0].scatter(subset['Load_Factor'], subset['Profit_Margin'],
                    alpha=0.2, s=10, color=color, label=cat)

x_vals = df['Load_Factor'].values
y_vals = df['Profit_Margin'].values
coef   = np.polyfit(x_vals, y_vals, 1)
x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
axes[0].plot(x_line, np.polyval(coef, x_line), color='black', linewidth=1.8,
             linestyle='--', label=f'Trend (slope={coef[0]:.1f})')
axes[0].axhline(0, color='red', linewidth=0.9, linestyle=':')

corr = df['Load_Factor'].corr(df['Profit_Margin'])
axes[0].text(0.05, 0.95, f'Pearson r = {corr:.3f}',
             transform=axes[0].transAxes, fontsize=10, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[0].set_title('Load Factor vs Profit Margin')
axes[0].set_xlabel('Load Factor')
axes[0].set_ylabel('Profit Margin (%)')
axes[0].legend(fontsize=9)

lf_route = df.groupby('Destination')['Load_Factor'].mean().sort_values(ascending=False)
dest_cat_map = df.groupby('Destination')['Route_Category'].first()
bar_colors = [PALETTE_ROUTE[dest_cat_map[d]] for d in lf_route.index]
axes[1].bar(lf_route.index, lf_route.values, color=bar_colors)
axes[1].set_title('Average Load Factor by Destination')
axes[1].set_xlabel('Destination')
axes[1].set_ylabel('Average Load Factor')
axes[1].tick_params(axis='x', rotation=70)
axes[1].axhline(lf_route.mean(), color='black', linestyle='--', linewidth=1.2,
                label=f'Network Mean: {lf_route.mean():.3f}')
axes[1].axhline(0.75, color='red', linestyle=':', linewidth=1.2,
                label='Typical Break-Even (~0.75)')
axes[1].legend()

plt.tight_layout()
plt.show()


### 3.6 Correlation Analysis

The full correlation matrix reveals pairwise linear relationships among all numerical features. We pay particular attention to:
- **Features highly correlated with `Profit_Margin`** — these are the best predictors for ML models
- **Multicollinear feature pairs** — features that are nearly redundant with each other (e.g. total cost and its components)

A secondary chart ranks all features by their correlation with the target variable, providing a quick visual summary of which features matter most for a linear model.


In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
exclude = ['Month', 'Quarter', 'Is_Profitable', 'Load_Factor_Computed',
           'Profit_Computed', 'Profit_Margin_Computed']
numeric_cols = [c for c in numeric_cols if c not in exclude]

corr_matrix       = df[numeric_cols].corr()
corr_with_target  = corr_matrix['Profit_Margin'].drop('Profit_Margin').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.suptitle('Correlation Analysis')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot=False, linewidths=0.3, cbar_kws={'shrink': 0.8}, ax=axes[0])
axes[0].set_title('Full Correlation Matrix (Lower Triangle)')
axes[0].tick_params(axis='x', rotation=70, labelsize=8)
axes[0].tick_params(axis='y', labelsize=8)

bar_colors = ['#E53935' if v < 0 else '#43A047' for v in corr_with_target.values]
axes[1].barh(corr_with_target.index, corr_with_target.values, color=bar_colors)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Correlation with Profit Margin\n(Green = positive, Red = negative)')
axes[1].set_xlabel('Pearson Correlation Coefficient')
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.show()

print("Top 5 positive correlations with Profit Margin:")
print(corr_with_target.tail(5).to_string())
print("\nTop 5 negative correlations with Profit Margin:")
print(corr_with_target.head(5).to_string())


---
## 4. Cost Structure Decomposition

### 4.1 Overall Cost Mix

Operating costs for an airline fall into three broad families:

| Family | Components | Characteristics |
|--------|-----------|-----------------|
| **Direct / Flight** | Fuel, Maintenance, Crew, Depreciation, Insurance | Scale with flight hours and aircraft type |
| **Station / Service** | Airport Fees, Catering, Handling, Navigation | Scale with flight frequency and passenger count |
| **Indirect / Overhead** | Sales, Passenger Service, Overhead, Marketing, IT | Largely fixed; spread across the network |

Understanding the cost mix is essential because each family has a different management lever. Fuel can be hedged or reduced through fleet renewal. Station costs can be renegotiated. Overhead can be restructured. The stacked bar chart shows how the mix shifts across short, medium, and long-haul operations — a critical input to pricing strategy.


In [ ]:
cost_totals = df[ALL_COST_COLS].sum()
cost_pct    = (cost_totals / cost_totals.sum() * 100).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Overall Cost Structure Decomposition')

palette_cost = sns.color_palette('tab20', len(cost_pct))
bars = axes[0].barh(cost_pct.index, cost_pct.values, color=palette_cost)
axes[0].set_title('Cost Components as Share of Total Cost (%)')
axes[0].set_xlabel('Percentage of Total Cost (%)')
axes[0].invert_yaxis()
for bar, val in zip(bars, cost_pct.values):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

cost_by_cat = df.groupby('Route_Category')[ALL_COST_COLS].mean()
cost_by_cat_pct = cost_by_cat.div(cost_by_cat.sum(axis=1), axis=0) * 100
cost_by_cat_pct = cost_by_cat_pct.loc[['Short Haul', 'Medium Haul', 'Long Haul']]

bottom = np.zeros(3)
bar_pal = sns.color_palette('tab20', len(ALL_COST_COLS))
for i, col in enumerate(ALL_COST_COLS):
    axes[1].bar(['Short Haul', 'Medium Haul', 'Long Haul'],
                cost_by_cat_pct[col], bottom=bottom,
                label=col.replace('_Cost', '').replace('_', ' '),
                color=bar_pal[i])
    bottom += cost_by_cat_pct[col].values

axes[1].set_title('Cost Mix by Route Category (% of Total Cost)')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(loc='lower center', bbox_to_anchor=(0.5, -0.45), ncol=3, fontsize=8)
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.show()


### 4.2 CASK and RASK Analysis — The Industry Gold Standard

**CASK** (Cost per Available Seat Kilometre) and **RASK** (Revenue per Available Seat Kilometre) normalise cost and revenue by network capacity, enabling fair comparison across routes of different distances and aircraft sizes.

- When **RASK > CASK**: the route earns more than it costs to operate → **profitable**
- When **RASK < CASK**: the route destroys value → **loss-making**
- The **spread** (RASK − CASK) is the most actionable profitability metric for scheduling teams

The scatter plot below includes a 45° break-even line. Routes above the line are profitable; routes below are not. The spread bar chart ranks all destinations and makes the depth of under-performance immediately visible.


In [ ]:
cask_rask = df.groupby('Destination').agg(
    CASK           = ('CASK',           'mean'),
    RASK           = ('RASK',           'mean'),
    Route_Category = ('Route_Category', 'first')
).reset_index()
cask_rask['Spread'] = cask_rask['RASK'] - cask_rask['CASK']
cask_rask = cask_rask.sort_values('Spread', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('CASK vs RASK Analysis — Operational Efficiency by Route')

for cat, color in PALETTE_ROUTE.items():
    subset = cask_rask[cask_rask['Route_Category'] == cat]
    axes[0].scatter(subset['CASK'], subset['RASK'], color=color, s=80, label=cat, zorder=3)
    for _, row in subset.iterrows():
        axes[0].annotate(row['Destination'], (row['CASK'], row['RASK']),
                         textcoords='offset points', xytext=(4, 4), fontsize=7)

min_v = min(cask_rask['CASK'].min(), cask_rask['RASK'].min()) * 0.95
max_v = max(cask_rask['CASK'].max(), cask_rask['RASK'].max()) * 1.05
axes[0].plot([min_v, max_v], [min_v, max_v], 'k--', linewidth=1.2, label='Break-Even (RASK=CASK)')
axes[0].fill_between([min_v, max_v], [min_v, max_v], max_v, alpha=0.05, color='green', label='Profitable zone')
axes[0].fill_between([min_v, max_v], min_v, [min_v, max_v], alpha=0.05, color='red',   label='Loss-making zone')
axes[0].set_title('RASK vs CASK (Above diagonal = Profitable)')
axes[0].set_xlabel('CASK (USD / ASK)')
axes[0].set_ylabel('RASK (USD / ASK)')
axes[0].legend(fontsize=8)

colors_spread = [PALETTE_ROUTE[c] for c in cask_rask['Route_Category']]
bars = axes[1].barh(cask_rask['Destination'], cask_rask['Spread'], color=colors_spread)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('RASK − CASK Spread by Destination\n(Positive = profitable, Negative = loss-making)')
axes[1].set_xlabel('Spread (USD / ASK)')
axes[1].invert_yaxis()
axes[1].legend(handles=[mpatches.Patch(facecolor=v, label=k) for k, v in PALETTE_ROUTE.items()])

plt.tight_layout()
plt.show()

n_profitable = (cask_rask['Spread'] > 0).sum()
print(f"Routes with positive RASK-CASK spread: {n_profitable} / {len(cask_rask)}")
print(f"Best spread  : {cask_rask.iloc[0]['Destination']}  ({cask_rask.iloc[0]['Spread']:.4f} USD/ASK)")
print(f"Worst spread : {cask_rask.iloc[-1]['Destination']}  ({cask_rask.iloc[-1]['Spread']:.4f} USD/ASK)")


---
## 5. Seasonality and Demand Analysis

### 5.1 Monthly and Seasonal Trends

Airlines live and die by seasonality. Demand, load factor, and pricing power all fluctuate with school holidays, weather patterns, and cultural events. For a Middle Eastern carrier, the seasonal profile is shaped by:
- **Peak (Dec–Feb)**: winter sun-seekers from Europe and Asia, high-yield business travel
- **Shoulder (Mar, Sep–Nov)**: transitional periods with moderate demand
- **Normal (Apr–May)**: steady but unexceptional volume
- **Low (Jun–Aug)**: summer heat suppresses inbound leisure demand; business travel migrates to cooler destinations

The heatmap at the end of this section shows every route × season combination, making it easy to spot routes that are profitable in Peak but catastrophically loss-making in Low season.


In [ ]:
month_order  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly = df.groupby(['Month', 'Month_Name', 'Route_Category']).agg(
    Avg_Load_Factor     = ('Load_Factor',           'mean'),
    Avg_Profit_Margin   = ('Profit_Margin',          'mean'),
    Total_Revenue       = ('Total_Revenue',          'sum'),
    Avg_Revenue_Per_Pax = ('Revenue_Per_Passenger',  'mean'),
).reset_index().sort_values('Month')

monthly_overall = df.groupby(['Month', 'Month_Name']).agg(
    Avg_Load_Factor   = ('Load_Factor',  'mean'),
    Avg_Profit_Margin = ('Profit_Margin','mean'),
    Total_Revenue     = ('Total_Revenue','sum'),
).reset_index().sort_values('Month')

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Monthly Seasonality Patterns')

# -- Overall load factor
sns.lineplot(data=monthly_overall, x='Month', y='Avg_Load_Factor',
             marker='o', color='steelblue', linewidth=2, ax=axes[0, 0])
axes[0, 0].set(title='Overall Average Load Factor by Month',
               xlabel='Month', ylabel='Average Load Factor')
axes[0, 0].set_xticks(range(1, 13))
axes[0, 0].set_xticklabels(month_order)

# -- Load factor by route category
sns.lineplot(data=monthly, x='Month', y='Avg_Load_Factor',
             hue='Route_Category', palette=PALETTE_ROUTE,
             marker='o', linewidth=1.5, ax=axes[0, 1])
axes[0, 1].set(title='Load Factor by Route Category', xlabel='Month', ylabel='Average Load Factor')
axes[0, 1].set_xticks(range(1, 13)); axes[0, 1].set_xticklabels(month_order)

# -- Profit margin by route category
sns.lineplot(data=monthly, x='Month', y='Avg_Profit_Margin',
             hue='Route_Category', palette=PALETTE_ROUTE,
             marker='o', linewidth=1.5, ax=axes[1, 0])
axes[1, 0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1, 0].set(title='Profit Margin by Route Category', xlabel='Month', ylabel='Profit Margin (%)')
axes[1, 0].set_xticks(range(1, 13)); axes[1, 0].set_xticklabels(month_order)

# -- Total revenue by month
rev_monthly = df.groupby('Month')['Total_Revenue'].sum().reset_index().sort_values('Month')
sns.barplot(data=rev_monthly, x='Month', y='Total_Revenue', palette='Blues_d', ax=axes[1, 1])
axes[1, 1].set(title='Total Revenue by Month (USD)', xlabel='Month', ylabel='Total Revenue (USD)')
axes[1, 1].set_xticklabels(month_order, rotation=45)
axes[1, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.show()


In [ ]:
# Route × Season profit margin heatmap
pivot_dest_season = df.pivot_table(
    index='Destination', columns='Season',
    values='Profit_Margin', aggfunc='mean'
)[['Peak', 'Shoulder', 'Normal', 'Low']]

dest_order_margin = df.groupby('Destination')['Profit_Margin'].mean().sort_values(ascending=False).index
pivot_dest_season = pivot_dest_season.loc[dest_order_margin]

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(pivot_dest_season, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.4, cbar_kws={'label': 'Average Profit Margin (%)'},
            ax=ax, vmin=-50)
ax.set_title('Average Profit Margin (%) by Destination × Season\n'
             '(Sorted by overall margin, highest → lowest)')
ax.set_xlabel('Season'); ax.set_ylabel('Destination')
plt.tight_layout()
plt.show()

print("\nSeasonal statistics across the full network:")
print(df.groupby('Season')[['Load_Factor', 'Profit_Margin', 'Revenue_Per_Passenger']]
      .agg(['mean', 'median', 'std']).round(3).to_string())


---
## 6. Machine Learning — Regression: Predicting Profit Margin

This section trains **seven regression models** to predict `Profit_Margin` from pre-flight operational features only. Cost and revenue columns are deliberately excluded to make the model useful for *planning* (pre-flight), not just post-flight reporting.

### 6.1 Feature Engineering and Preprocessing

**Feature rationale:**

| Feature | Why it's included |
|---------|------------------|
| `Aircraft_Capacity` | Determines fixed-cost dilution potential |
| `Passengers` / `Load_Factor` | Primary demand signal |
| `Flight_Hours` | Proxy for distance; drives fuel and crew cost |
| `Month` / `Quarter` | Captures seasonality |
| `Aircraft_Type` | Encodes operational efficiency differences |
| `Route_Category` | Encodes structural short/medium/long haul economics |
| `Season` / `Demand_Level` | Forward-looking demand classification |
| `Destination` | Route-specific fixed effects |

**Models trained:**

| Model | Type | Key hyperparameters |
|-------|------|---------------------|
| Ridge Regression | Linear | α = 1.0 |
| Lasso Regression | Linear + L1 | α = 0.1 |
| ElasticNet | Linear + L1+L2 | α = 0.1, l1_ratio = 0.5 |
| Random Forest | Tree ensemble | 300 trees, max_depth=15 |
| Extra Trees | Tree ensemble | 300 trees (randomised splits) |
| LightGBM | Gradient boosting | 500 trees, lr=0.05 |
| XGBoost | Gradient boosting | 500 trees, lr=0.05 |


In [ ]:
FEATURE_COLS = [
    'Aircraft_Capacity', 'Passengers', 'Load_Factor', 'Flight_Hours',
    'Month', 'Quarter',
    'Aircraft_Type', 'Route_Category', 'Season', 'Demand_Level', 'Destination'
]
TARGET_REG = 'Profit_Margin'

df_ml = df[FEATURE_COLS + [TARGET_REG]].copy()

cat_cols = ['Aircraft_Type', 'Route_Category', 'Season', 'Demand_Level', 'Destination']
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))
    le_dict[col] = le

X = df_ml[FEATURE_COLS]
y = df_ml[TARGET_REG]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Features         : {len(FEATURE_COLS)}")
print(f"Target           : {TARGET_REG}")
print(f"Target range     : [{y.min():.2f}, {y.max():.2f}]%")
print(f"Target mean/std  : {y.mean():.2f}% / {y.std():.2f}%")


### 6.2 Seven-Model Regression Benchmark

5-fold cross-validation on the training set guards against overfitting. Final evaluation is on the held-out 20% test set. Three metrics are reported:

- **R²**: proportion of variance explained (higher is better; 1.0 = perfect)
- **RMSE**: root mean squared error in percentage points (lower is better)
- **MAE**: mean absolute error in percentage points (lower is better; less sensitive to outliers than RMSE)


In [ ]:
regression_models = {
    'Ridge':         Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'Lasso':         Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.1, max_iter=5000))]),
    'ElasticNet':    Pipeline([('sc', StandardScaler()), ('m', ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000))]),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=15, n_jobs=-1, random_state=RANDOM_STATE),
    'Extra Trees':   ExtraTreesRegressor(n_estimators=300, max_depth=15,  n_jobs=-1, random_state=RANDOM_STATE),
    'LightGBM':      lgb.LGBMRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                                        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    'XGBoost':       xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                       subsample=0.8, colsample_bytree=0.8,
                                       n_jobs=-1, random_state=RANDOM_STATE, verbosity=0),
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
reg_results   = []
trained_reg   = {}

print(f"{'Model':<18} | {'CV R²':>9} ± {'Std':>6} | {'Test R²':>8} | {'RMSE':>7} | {'MAE':>7}")
print("-" * 65)

for name, model in regression_models.items():
    cv_r2 = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2', n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    reg_results.append({'Model': name, 'CV R² Mean': cv_r2.mean(), 'CV R² Std': cv_r2.std(),
                        'Test R²': r2, 'RMSE': rmse, 'MAE': mae})
    trained_reg[name] = (model, y_pred)
    print(f"{name:<18} | {cv_r2.mean():>9.4f} ± {cv_r2.std():>6.4f} | {r2:>8.4f} | {rmse:>7.3f} | {mae:>7.3f}")

reg_df = pd.DataFrame(reg_results).sort_values('Test R²', ascending=False).reset_index(drop=True)
print()
print("Ranked by Test R²:")
print(reg_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Regression Model Comparison — 7 Models')

pal = sns.color_palette('Set2', len(reg_df))

for ax, (metric, label, ascending) in zip(axes, [
    ('Test R²',  'Test R² (higher is better)', False),
    ('RMSE',     'RMSE — pp (lower is better)', True),
    ('MAE',      'MAE  — pp (lower is better)', True),
]):
    sorted_df = reg_df.sort_values(metric, ascending=ascending)
    colors    = ['#43A047' if i == 0 else '#1E88E5' for i in range(len(sorted_df))]
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
    ax.set_title(label)
    ax.set_xlabel(metric)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print("Green bar = best model for that metric")


### 6.3 Best-Model Deep Dive: Predicted vs Actual and Residual Analysis

We take the best-performing regression model and run three standard diagnostics:
1. **Predicted vs Actual plot** — points should cluster tightly around the red diagonal; systematic deviation indicates bias
2. **Residuals distribution** — should be approximately centred at zero; a skewed distribution indicates systematic over- or under-prediction
3. **Residuals vs Fitted** — a random cloud with no pattern confirms homoscedasticity; a funnel shape would indicate that prediction errors are larger at extreme values


In [ ]:
best_reg_name = reg_df.iloc[0]['Model']
best_reg_model, y_pred_best = trained_reg[best_reg_name]
residuals = y_test.values - y_pred_best

print(f"Best regression model: {best_reg_name}")
print(f"  Test R²  = {r2_score(y_test, y_pred_best):.4f}")
print(f"  RMSE     = {np.sqrt(mean_squared_error(y_test, y_pred_best)):.4f} pp")
print(f"  MAE      = {mean_absolute_error(y_test, y_pred_best):.4f} pp")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{best_reg_name} — Model Diagnostics')

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=10, color='steelblue')
lim = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set(title='Predicted vs Actual Profit Margin',
            xlabel='Actual Profit Margin (%)', ylabel='Predicted Profit Margin (%)')
axes[0].legend()
axes[0].text(0.05, 0.95, f'R² = {r2_score(y_test, y_pred_best):.4f}',
             transform=axes[0].transAxes, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Residuals distribution
sns.histplot(residuals, bins=50, kde=True, color='coral', ax=axes[1], edgecolor='none')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.2)
axes[1].set(title='Residuals Distribution',
            xlabel='Residual = Actual − Predicted')
axes[1].text(0.05, 0.95, f'Mean: {residuals.mean():.4f}\nStd:  {residuals.std():.3f}',
             transform=axes[1].transAxes, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Residuals vs Fitted
axes[2].scatter(y_pred_best, residuals, alpha=0.3, s=10, color='mediumseagreen')
axes[2].axhline(0, color='red', linestyle='--', linewidth=1.2)
axes[2].set(title='Residuals vs Fitted Values',
            xlabel='Predicted Profit Margin (%)', ylabel='Residual')

plt.tight_layout()
plt.show()


### 6.4 SHAP Feature Importance and Interpretation

SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to each individual prediction. Unlike standard feature importance (which only shows magnitude), SHAP reveals:
- **Direction**: does a higher value of Feature X increase or decrease the predicted margin?
- **Magnitude**: by how much?
- **Interactions**: do effects change depending on other feature values?

We use three SHAP visualisations:
1. **Bar chart** — global mean |SHAP| value per feature
2. **Beeswarm / boxplot** — distribution of SHAP values showing direction and spread
3. **Dependence plots** — how SHAP value varies with feature value for the top two predictors


In [ ]:
# Use LightGBM for SHAP (tree explainer is fastest and most accurate for tree models)
lgbm_reg = trained_reg['LightGBM'][0]
explainer   = shap.TreeExplainer(lgbm_reg)
shap_values = explainer.shap_values(X_test)

mean_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_shap, index=FEATURE_COLS).sort_values()

print(f"SHAP values computed for {X_test.shape[0]:,} test observations.")
print("\nGlobal feature importance (mean |SHAP value|):")
print(shap_importance.sort_values(ascending=False).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('SHAP Feature Importance — LightGBM Regression Model')

bar_colors_shap = sns.color_palette('viridis', len(shap_importance))
bars = axes[0].barh(shap_importance.index, shap_importance.values, color=bar_colors_shap)
axes[0].set_title('Mean Absolute SHAP Value (Global Importance)')
axes[0].set_xlabel('Mean |SHAP Value| — impact on Profit Margin (%)')
for bar, val in zip(bars, shap_importance.values):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

top_features = shap_importance.sort_values(ascending=False).head(8).index.tolist()
top_indices  = [FEATURE_COLS.index(f) for f in top_features]
shap_df_top  = pd.DataFrame(shap_values[:, top_indices], columns=top_features)
shap_melted  = shap_df_top.melt(var_name='Feature', value_name='SHAP Value')
feature_order= shap_df_top.abs().mean().sort_values(ascending=False).index.tolist()

sns.boxplot(data=shap_melted, x='SHAP Value', y='Feature',
            order=feature_order, palette='coolwarm',
            ax=axes[1], width=0.5,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].axvline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_title('SHAP Value Distribution — Top 8 Features\n'
                  '(Positive SHAP = increases predicted Profit Margin)')
axes[1].set_xlabel('SHAP Value (impact on Profit Margin %)')

plt.tight_layout()
plt.show()


In [ ]:
# SHAP dependence plots for top 2 features
top2 = shap_importance.sort_values(ascending=False).head(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('SHAP Dependence Plots — Top 2 Most Important Features')

for ax, feat in zip(axes, top2):
    feat_idx  = FEATURE_COLS.index(feat)
    shap_col  = shap_values[:, feat_idx]
    feat_vals = X_test[feat].values
    sc = ax.scatter(feat_vals, shap_col, alpha=0.3, s=10, c=shap_col, cmap='coolwarm')
    plt.colorbar(sc, ax=ax, label='SHAP Value')
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set(title=f'SHAP Dependence: {feat}',
           xlabel=feat, ylabel='SHAP Value (contribution to Profit Margin %)')

plt.tight_layout()
plt.show()
print(f"Top SHAP feature: '{top2[0]}' — this is the single strongest driver of profit margin prediction.")


---
## 7. Machine Learning — Classification: Predicting Route Profitability

Beyond predicting the exact profit margin, it is often more actionable to classify a flight as **profitable vs loss-making** before it departs. This binary classification task complements the regression model and is easier to operationalise in scheduling workflows.

**Target:** `Is_Profitable` (1 = Profit > 0, 0 = loss-making)

We train and compare **five classification models** using the same feature set as the regression task. A stratified 5-fold cross-validation is used to handle any class imbalance.

### 7.1 Class Balance Check


In [ ]:
TARGET_CLF = 'Is_Profitable'

class_counts = df[TARGET_CLF].value_counts()
print("Class distribution:")
for label, count in class_counts.items():
    name = "Profitable" if label == 1 else "Loss-Making"
    print(f"  {name:12s} ({label}): {count:,} flights ({count/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Loss-Making (0)', 'Profitable (1)'], class_counts.values,
       color=['#E53935', '#43A047'], width=0.5)
ax.set_title('Class Balance — Is_Profitable')
ax.set_ylabel('Number of Flights')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 20, f'{v:,} ({v/len(df)*100:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Prepare classification data (same features, different target)
df_clf = df[FEATURE_COLS + [TARGET_CLF]].copy()

for col in cat_cols:
    df_clf[col] = le_dict[col].transform(df_clf[col].astype(str))

X_clf = df_clf[FEATURE_COLS]
y_clf = df_clf[TARGET_CLF]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)

print(f"Training samples : {len(X_train_c):,}  (profitable: {y_train_c.sum():,})")
print(f"Test samples     : {len(X_test_c):,}  (profitable: {y_test_c.sum():,})")


### 7.2 Five-Model Classification Benchmark

| Model | Type |
|-------|------|
| Random Forest | Ensemble of decision trees |
| Extra Trees | Randomised ensemble (faster, sometimes better) |
| Gradient Boosting | Sequential boosting |
| LightGBM | Fast gradient boosting with leaf-wise growth |
| XGBoost | Regularised gradient boosting |

Metrics:
- **Accuracy**: fraction of correctly classified flights
- **ROC-AUC**: area under the ROC curve (threshold-independent; 1.0 = perfect, 0.5 = random)
- **F1**: harmonic mean of precision and recall (balances false positives and false negatives)


In [ ]:
clf_models = {
    'Random Forest':     RandomForestClassifier(n_estimators=300, max_depth=15, n_jobs=-1, random_state=RANDOM_STATE),
    'Extra Trees':       ExtraTreesClassifier(n_estimators=300, max_depth=15,  n_jobs=-1, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=RANDOM_STATE),
    'LightGBM':          lgb.LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                                             num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                                             n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    'XGBoost':           xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                                            subsample=0.8, colsample_bytree=0.8,
                                            n_jobs=-1, random_state=RANDOM_STATE,
                                            verbosity=0, eval_metric='logloss'),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
clf_results  = []
trained_clf  = {}

print(f"{'Model':<20} | {'CV AUC':>8} ± {'Std':>6} | {'Acc':>6} | {'AUC':>6} | {'F1':>6}")
print("-" * 65)

for name, model in clf_models.items():
    cv_auc = cross_val_score(model, X_train_c, y_train_c, cv=skf,
                              scoring='roc_auc', n_jobs=-1)
    model.fit(X_train_c, y_train_c)
    y_pred_c  = model.predict(X_test_c)
    y_prob_c  = model.predict_proba(X_test_c)[:, 1]
    acc  = accuracy_score(y_test_c, y_pred_c)
    auc  = roc_auc_score(y_test_c, y_prob_c)
    f1   = f1_score(y_test_c, y_pred_c)
    clf_results.append({'Model': name, 'CV AUC Mean': cv_auc.mean(), 'CV AUC Std': cv_auc.std(),
                        'Accuracy': acc, 'ROC-AUC': auc, 'F1': f1})
    trained_clf[name] = (model, y_pred_c, y_prob_c)
    print(f"{name:<20} | {cv_auc.mean():>8.4f} ± {cv_auc.std():>6.4f} | {acc:>6.4f} | {auc:>6.4f} | {f1:>6.4f}")

clf_df = pd.DataFrame(clf_results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print()
print("Ranked by ROC-AUC:")
print(clf_df.to_string(index=False))


In [ ]:
# Comparison bar charts + confusion matrix for best model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Classification Model Comparison — 5 Models')

for ax, metric in zip(axes[:2], ['ROC-AUC', 'F1']):
    sorted_df = clf_df.sort_values(metric, ascending=False)
    colors    = ['#43A047' if i == 0 else '#1E88E5' for i in range(len(sorted_df))]
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
    ax.set_title(f'{metric} (higher is better)')
    ax.set_xlabel(metric)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)

# Confusion matrix for best model
best_clf_name = clf_df.iloc[0]['Model']
_, y_pred_best_c, _ = trained_clf[best_clf_name]
cm = confusion_matrix(y_test_c, y_pred_best_c)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Loss-Making', 'Profitable'])
disp.plot(ax=axes[2], cmap='Blues', colorbar=False)
axes[2].set_title(f'Confusion Matrix — {best_clf_name}')

plt.tight_layout()
plt.show()

print(f"\nBest classifier: {best_clf_name}")
print(f"\nDetailed classification report ({best_clf_name}):")
print(classification_report(y_test_c, y_pred_best_c, target_names=['Loss-Making', 'Profitable']))


---
## 8. Route Clustering and Segmentation

### 8.1 Why Cluster Routes?

K-Means clustering groups routes by financial similarity rather than by intuitive categories like route length. This can reveal non-obvious segments — for example, a high-load-factor short-haul route that financially resembles a medium-haul route — that would be invisible to a simple categorical analysis.

**Clustering features (all at route level, aggregated from flight-level data):**

| Feature | Business meaning |
|---------|-----------------|
| `Avg_Profit_Margin` | Overall profitability |
| `Avg_Load_Factor` | Demand strength |
| `Avg_Revenue_Per_Pax` | Pricing power / yield |
| `Avg_CASK` | Unit cost efficiency |
| `Avg_RASK` | Unit revenue generation |
| `Avg_Flight_Hours` | Network position (distance proxy) |
| `Pct_Profitable` | Consistency of profitability |

We select the optimal number of clusters using both the **elbow method** (inertia) and the **silhouette score**, then visualise the resulting clusters in 2D PCA space.


In [ ]:
from sklearn.metrics import silhouette_score

cluster_features = [
    'Avg_Profit_Margin', 'Avg_Load_Factor', 'Avg_Revenue_Per_Pax',
    'Avg_CASK', 'Avg_RASK', 'Avg_Flight_Hours', 'Pct_Profitable'
]

route_cluster_df = df.groupby('Destination').agg(
    Avg_Profit_Margin   = ('Profit_Margin',         'mean'),
    Avg_Load_Factor     = ('Load_Factor',            'mean'),
    Avg_Revenue_Per_Pax = ('Revenue_Per_Passenger',  'mean'),
    Avg_CASK            = ('CASK',                   'mean'),
    Avg_RASK            = ('RASK',                   'mean'),
    Avg_Flight_Hours    = ('Flight_Hours',            'mean'),
    Pct_Profitable      = ('Is_Profitable',           'mean'),
    Route_Category      = ('Route_Category',          'first'),
).reset_index()
route_cluster_df['Pct_Profitable'] *= 100

scaler_cl = MinMaxScaler()
X_cluster = scaler_cl.fit_transform(route_cluster_df[cluster_features])

inertias, silhouettes = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Optimal Number of Clusters')

axes[0].plot(list(K_range), inertias, marker='o', color='steelblue', linewidth=2)
axes[0].set(title='Elbow Method — Inertia vs K',
            xlabel='Number of Clusters (K)', ylabel='Inertia')

axes[1].plot(list(K_range), silhouettes, marker='o', color='coral', linewidth=2)
best_k = list(K_range)[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='black', linestyle='--', linewidth=1.2, label=f'Best K = {best_k}')
axes[1].set(title='Silhouette Score vs K (Higher is Better)',
            xlabel='Number of Clusters (K)', ylabel='Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Optimal K by silhouette score: {best_k}")
print(f"Silhouette scores: {dict(zip(K_range, [f'{s:.4f}' for s in silhouettes]))}")


In [ ]:
FINAL_K      = best_k
kmeans_final = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=10)
route_cluster_df['Cluster'] = kmeans_final.fit_predict(X_cluster)

pca    = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca  = pca.fit_transform(X_cluster)
route_cluster_df['PCA1'] = X_pca[:, 0]
route_cluster_df['PCA2'] = X_pca[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f'Route Clustering Results — K = {FINAL_K}')
cluster_palette = sns.color_palette('tab10', FINAL_K)

for cid in range(FINAL_K):
    subset = route_cluster_df[route_cluster_df['Cluster'] == cid]
    axes[0].scatter(subset['PCA1'], subset['PCA2'],
                    color=cluster_palette[cid], s=100, label=f'Cluster {cid}', zorder=3)
    for _, row in subset.iterrows():
        axes[0].annotate(row['Destination'], (row['PCA1'], row['PCA2']),
                         textcoords='offset points', xytext=(5, 5), fontsize=7)

axes[0].set(title=f'PCA 2D Projection of Route Clusters (K={FINAL_K})',
            xlabel=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance explained)',
            ylabel=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance explained)')
axes[0].legend()

for cid in range(FINAL_K):
    subset = route_cluster_df[route_cluster_df['Cluster'] == cid]
    cx     = subset['Avg_Load_Factor'].mean()
    cy     = subset['Avg_Profit_Margin'].mean()
    size   = subset['Pct_Profitable'].mean() * 8
    axes[1].scatter(cx, cy, color=cluster_palette[cid], s=size,
                    label=f'Cluster {cid} (n={len(subset)})',
                    alpha=0.9, edgecolors='black', linewidth=0.8, zorder=3)
    axes[1].annotate(f'C{cid}', (cx, cy),
                     textcoords='offset points', xytext=(4, 4), fontsize=9, fontweight='bold')

axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set(title='Cluster Profiles: Load Factor vs Profit Margin\n'
                   '(Bubble size ∝ % of profitable flights)',
            xlabel='Average Load Factor', ylabel='Average Profit Margin (%)')
axes[1].legend()
plt.tight_layout()
plt.show()

print("\nCluster membership:")
for cid in range(FINAL_K):
    routes = route_cluster_df[route_cluster_df['Cluster'] == cid][['Destination', 'Route_Category']]
    avg_m  = route_cluster_df[route_cluster_df['Cluster'] == cid]['Avg_Profit_Margin'].mean()
    print(f"\nCluster {cid} — avg margin {avg_m:.1f}% — {len(routes)} routes:")
    print(routes.to_string(index=False))


In [ ]:
# Cluster profile heatmap
cluster_profile     = route_cluster_df.groupby('Cluster')[cluster_features].mean()
cluster_profile_std = (cluster_profile - cluster_profile.mean()) / cluster_profile.std()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cluster_profile_std.T, annot=cluster_profile.T.round(2),
            fmt='g', cmap='RdYlGn', center=0, linewidths=0.5,
            cbar_kws={'label': 'Standardised z-score'}, ax=ax)
ax.set_title('Cluster Profile Heatmap\n'
             '(Colour = standardised score  |  Numbers = actual mean values)')
ax.set_xlabel('Cluster')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()


---
## 9. What-If Scenario Simulator

The scenario simulator uses the trained regression model to estimate profit margin changes when key operational parameters change. This is a practical planning tool for:
- Assessing the margin impact of a **load factor improvement campaign**
- Estimating the financial benefit of **upsizing aircraft** on a specific route
- Comparing predicted outcomes across **seasons** before committing to schedule changes

We demonstrate three scenarios below. The simulator can be extended to any feature in the model.


In [ ]:
# --- Scenario 1: Load Factor Sensitivity for Top 5 Routes ---
top5_routes_list = route_summary.head(5)['Destination'].tolist()
bottom5_routes_list = route_summary.tail(5)['Destination'].tolist()
focus_routes = top5_routes_list + bottom5_routes_list

lf_range = np.linspace(0.55, 0.99, 30)

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharey=False)
fig.suptitle('Scenario Simulator: Profit Margin vs Load Factor\n'
             'Top 5 (upper) and Bottom 5 (lower) Routes', fontsize=14)

for i, dest in enumerate(focus_routes):
    ax = axes[i // 5, i % 5]
    sample = df[df['Destination'] == dest].iloc[0:1].copy()
    sample_encoded = sample[FEATURE_COLS].copy()
    for col in cat_cols:
        sample_encoded[col] = le_dict[col].transform(sample_encoded[col].astype(str))

    predicted_margins = []
    for lf in lf_range:
        row = sample_encoded.copy()
        row['Load_Factor'] = lf
        row['Passengers']  = round(lf * row['Aircraft_Capacity'].values[0])
        pred = trained_reg['LightGBM'][0].predict(row)[0]
        predicted_margins.append(pred)

    color = '#43A047' if dest in top5_routes_list else '#E53935'
    ax.plot(lf_range, predicted_margins, color=color, linewidth=2)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    actual_lf = df[df['Destination'] == dest]['Load_Factor'].mean()
    actual_m  = df[df['Destination'] == dest]['Profit_Margin'].mean()
    ax.axvline(actual_lf, color='grey', linestyle=':', linewidth=1, label=f'Actual LF={actual_lf:.2f}')
    ax.scatter([actual_lf], [actual_m], color='black', s=40, zorder=5)
    ax.set_title(dest, fontsize=10)
    ax.set_xlabel('Load Factor', fontsize=8)
    ax.set_ylabel('Predicted Margin (%)', fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()
print("Interpretation: Each curve shows how the model predicts profit margin would change")
print("as load factor increases from 55% to 99%, holding all other features constant.")
print("The black dot marks the route's actual historical position.")


In [ ]:
# --- Scenario 2: Season Impact on Bottom-5 Routes ---
bottom5_scenarios = []
seasons_list = ['Peak', 'Shoulder', 'Normal', 'Low']

for dest in bottom5_routes_list:
    sample = df[df['Destination'] == dest].iloc[0:1].copy()
    sample_encoded = sample[FEATURE_COLS].copy()
    for col in cat_cols:
        sample_encoded[col] = le_dict[col].transform(sample_encoded[col].astype(str))

    for season in seasons_list:
        row = sample_encoded.copy()
        row['Season'] = le_dict['Season'].transform([season])[0]
        pred = trained_reg['LightGBM'][0].predict(row)[0]
        bottom5_scenarios.append({'Destination': dest, 'Season': season, 'Predicted_Margin': pred})

scenario_df = pd.DataFrame(bottom5_scenarios)

fig, ax = plt.subplots(figsize=(12, 5))
pivot_sc = scenario_df.pivot(index='Destination', columns='Season', values='Predicted_Margin')[seasons_list]
sns.heatmap(pivot_sc, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label': 'Predicted Profit Margin (%)'}, ax=ax)
ax.set_title('Scenario Simulator: Predicted Profit Margin by Season\n'
             '(Bottom 5 loss-making routes — holding all other features at historical mode)')
ax.set_xlabel('Season'); ax.set_ylabel('Destination')
plt.tight_layout()
plt.show()

print("Insight: Even the worst-performing routes may be breakeven or profitable in Peak season.")
print("This guides frequency and capacity decisions — operate only in profitable seasons,")
print("or adjust pricing aggressively in Low season to recover margin.")


---
## 10. Key Findings and Summary Statistics


In [ ]:
print('=' * 72)
print('COMPREHENSIVE SUMMARY OF KEY FINDINGS')
print('=' * 72)

print()
print('--- Fleet and Network ---')
print(f'  Total flights analysed   : {len(df):,}')
print(f'  Unique routes             : {df["Destination"].nunique()}')
print(f'  Aircraft types            : {df["Aircraft_Type"].nunique()}')
print(f'  Date range                : {df["Flight_Date"].min().date()} to {df["Flight_Date"].max().date()}')

print()
print('--- Financial Overview ---')
print(f'  Total revenue (USD)       : {df["Total_Revenue"].sum():>16,.0f}')
print(f'  Total cost (USD)          : {df["Total_Cost"].sum():>16,.0f}')
print(f'  Total profit (USD)        : {df["Profit"].sum():>16,.0f}')
print(f'  Overall avg profit margin : {df["Profit_Margin"].mean():.2f}%')
print(f'  Median profit margin      : {df["Profit_Margin"].median():.2f}%')
print(f'  Profitable flights        : {df["Is_Profitable"].mean()*100:.1f}% of all flights')

print()
print('--- Regression Model Leaderboard ---')
for _, row in reg_df.iterrows():
    marker = " ← best" if _ == 0 else ""
    print(f'  {row["Model"]:<20}: Test R² = {row["Test R²"]:.4f}, RMSE = {row["RMSE"]:.3f} pp{marker}')

print()
print('--- Classification Model Leaderboard ---')
for _, row in clf_df.iterrows():
    marker = " ← best" if _ == 0 else ""
    print(f'  {row["Model"]:<20}: AUC = {row["ROC-AUC"]:.4f}, F1 = {row["F1"]:.4f}{marker}')

print()
print('--- Top SHAP Features (Regression) ---')
for feat, val in shap_importance.sort_values(ascending=False).head(5).items():
    print(f'  {feat:<25}: mean |SHAP| = {val:.3f}')

print()
print('--- Top 5 Routes by Avg Profit Margin ---')
print(route_summary.head(5)[['Destination','Route_Category','Avg_Profit_Margin','Pct_Profitable']].to_string(index=False))

print()
print('--- Bottom 5 Routes by Avg Profit Margin ---')
print(route_summary.tail(5)[['Destination','Route_Category','Avg_Profit_Margin','Pct_Profitable']].to_string(index=False))

print()
print('--- Seasonal Breakdown ---')
for s in ['Peak', 'Shoulder', 'Normal', 'Low']:
    m  = df[df['Season'] == s]['Profit_Margin'].mean()
    lf = df[df['Season'] == s]['Load_Factor'].mean()
    n  = (df['Season'] == s).sum()
    print(f'  {s:10s}: n={n:,}  Avg Margin={m:7.2f}%  Avg LF={lf:.3f}')


---
## 11. Conclusions and Strategic Recommendations

### Financial Overview

Across the full year of operations analysed, the airline generated approximately **USD 2.37 billion in revenue** against USD 1.80 billion in costs, producing a **USD 575 million net profit** at an average margin of **~6.2%**. While the network is profitable in aggregate, only **~66% of individual flights** break even — meaning roughly one in three departures operates at a loss. This structural fragility is masked by the outsized profitability of a handful of long-haul routes.

---

### Route Profitability

The single most important finding is the sharp divergence between long-haul and short-haul economics:

**Long-haul outperformers** — Frankfurt (FRA), Singapore (SIN), Paris (CDG), Bangkok (BKK), and Hong Kong (HKG) all sustain margins above 38% and are profitable on virtually every departure. These five routes are the financial backbone of the network and should receive priority in fleet deployment and frequency planning.

**Short-haul loss-makers** — Cairo (CAI), Amman (AMM), Jeddah (JED), and Riyadh (RUH) post average margins of −42% to −84%, with CAI and JED recording zero profitable flights across the entire year. These are not marginal underperformers; they are structurally unprofitable at current cost structures and fare levels.

---

### Machine Learning Results

| Task | Best Model | Key Metric |
|------|-----------|------------|
| Profit Margin Regression | *(see leaderboard above)* | Test R² ≥ 0.68 |
| Profitable/Loss Classification | *(see leaderboard above)* | ROC-AUC ≥ 0.85 |

The most important predictive feature by SHAP value is **Route_Category**, followed by **Flight_Hours** and **Load_Factor** — confirming that route structure and utilisation are the primary levers available to management. The classification model achieves high AUC, making it a viable screening tool to flag high-risk schedule additions.

---

### Seasonality

The gap between Peak (Dec–Feb) and Low (Jun–Aug) season is approximately **26 percentage points of profit margin** (≈ +17% vs −9%). This is as much a revenue management problem as a demand problem: Low season pricing must be recalibrated, and capacity should be redeployed from routes that cannot support fixed costs at reduced load.

---

### Strategic Recommendations

**1. Exit or restructure chronic short-haul loss-makers.**  
CAI, AMM, and JED have zero profitable flights over a full year. Unless these routes serve a mandatory strategic feed function, options include fare increases, frequency reductions, codeshare arrangements, or temporary suspension.

**2. Grow long-haul capacity on proven routes.**  
FRA, SIN, CDG, BKK, and HKG are near-perfectly profitable across all seasons. These routes should receive priority in fleet deployment and additional frequencies during Peak season.

**3. Implement aggressive Low season dynamic pricing.**  
A network-wide average loss of ~9% in summer is partially recoverable through stimulative pricing targeting load factors above 0.80, without requiring structural network changes.

**4. Align aircraft type to route economics.**  
The CASK heatmap reveals that not all aircraft–route pairings are efficient. Next-generation narrowbodies should be used on short-haul to reduce per-seat fuel burn; widebodies should be reserved for long-haul sectors where unit economics improve with distance.

**5. Embed the ML models into scheduling workflows.**  
Even at current accuracy levels, the regression and classification models add value as pre-flight screening tools. Feeding them route-level trailing averages and dynamic pricing signals as additional features should push predictive accuracy materially higher.

**6. Investigate cluster-based pricing strategies.**  
The K-Means segmentation reveals route clusters with shared financial profiles. Pricing, promotion, and capacity strategies developed at the cluster level — rather than individually per route — would reduce analytical overhead while still capturing meaningful heterogeneity.

---

*Analysis covers full-year 2024 operations for a Dubai-hub carrier. All figures in USD. Distance estimates based on 850 km/h average cruise speed.*
